<a href="https://colab.research.google.com/github/AnPay/Templates/blob/main/Payal_Interview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training a RAG Re-ranker model

**Exercise:**
Assume you have a **RAG pipeline** that uses **single-step retrieval** (e.g. one embedding lookup over a chunk index, then the top chunks go to the generator). The RAG embedding model was not explicitly trained on chip-design data, so you want to train a re-ranker model to augment this RAG setup.

You have **some data**: some human annotated question–answer–chunk–document tuples, as well as a set of chip-design specific documents.
We want to:
1. Create a large dataset that combines both these datasets -- for questions in the annotated dataset, we now want to obtain training datapoints from the documents.
2. Define a loss-function to train a cross-encoder re-ranking model and train on the new full dataset

**Outline:**
1. **Discussion** — Conceptual questions about re-ranking models and traning them
2. **Data & task walkthrough** — Walk through the data and the task
4. **Dataset generation** — Implement chunking and create the training dataset
5. **Loss function** — Update the MSE loss to prevent minimize drift from the original model
6. **Run (Optional)** — Execute the full pipeline


In [ ]:
%pip install -q torch transformers datasets accelerate sentence-transformers


In [ ]:
## Imports
from __future__ import annotations

import random
import re
from dataclasses import dataclass
from typing import Literal, TypedDict, get_args

import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    PreTrainedModel,
    PreTrainedTokenizerBase,
    Trainer,
    TrainingArguments,
)

from typing import Callable, Optional, Literal, Dict, Any

## Section 1: Discussion (5 mins)

---

1. In a RAG system, where does a re-ranker sit relative to retrieval and the LLM? What problem does it solve that a vector DB + top‑k alone does not?

2. What is the difference between a bi-encoder and cross-encoder model?


3. When constructing a dataset, we want to build a "balanced" dataset with a distribution conducive to generalization. Describe why the following data-categories are needed. What is a hard-negative?

In [ ]:
RerankRegressionCategory = Literal["positive", "negative", "hard_negative"]

class RerankRegressionDataPoint(TypedDict):
    """One training row for the cross-encoder re-ranker."""
    question: str
    chunk: str
    similarity_score: float
    category: RerankRegressionCategory


## Section 2: Data & Task Walkthrough (10 mins)

---

### Walk through the data

You have access to:
1. **`ANNOTATED_DATA`** — human-annotated datapoints, each with: `question`, `answer`, `chunk` (the golden text-chunk), `document` (source document of the chunk).
2. **`CORPUS_DOCUMENTS`** — a list of multi-paragraph documents that form the retrieval corpus.

A small portion of each is provided below. In production there are ~100 annotated datapoints and ~5,000 corpus documents.


To train the re-ranker model, we will need to convert corpus documents into datapoints of the folllowing type

In [ ]:
class RerankRegressionDataPoint(TypedDict):
    """One training row for the cross-encoder re-ranker."""
    question: str
    chunk: str
    similarity_score: float
    category: str

In [ ]:
#@title Raw Annotated Data (Expand to look at raw data)


ANNOTATED_DATA: list[dict[str, str]] = [
    {
        "question": "What does $past capture in SystemVerilog assertions?",
        "answer": "The value of an expression at the previous clocking event.",
        "chunk": "In SVA, $past(expr) returns the sampled value of expr at the previous clocking event of the assertion.",
        "document": "SVA basics: Assertions are evaluated on clocking events. In SVA, $past(expr) returns the sampled value of expr at the previous clocking event of the assertion. Related: $rose and $fell detect edges."
    },

    {
        "question": "How is disable iff used in assertions?",
        "answer": "It disables the assertion when a condition is true.",
        "chunk": "The disable iff (reset) construct disables the entire assertion while reset is asserted, avoiding false failures during reset.",
        "document": "Assertion control: The disable iff (reset) construct disables the entire assertion while reset is asserted, avoiding false failures during reset. Properties can also be aborted with procedural code."
    },

    {
        "question": "What is the role of a bind statement in formal verification?",
        "answer": "It inserts monitor/checker modules without editing RTL.",
        "chunk": "Use bind to instantiate a checker or SVA module alongside RTL without modifying the RTL source files.",
        "document": "Formal setup: Use bind to instantiate a checker or SVA module alongside RTL without modifying the RTL source files. Jasper and similar tools elaborate the bound hierarchy."
    },

    {
        "question": "What does bounded proof depth mean in model checking?",
        "answer": "It only proves properties up to a fixed number of time steps.",
        "chunk": "Bounded model checking proves properties only up to a fixed depth N; deeper bugs may remain undiscovered.",
        "document": "Formal limits: Bounded model checking proves properties only up to a fixed depth N; deeper bugs may remain undiscovered. Increasing depth trades runtime for coverage of longer traces."
    },

    {
        "question": "In UVM, what is a virtual sequencer's typical role?",
        "answer": "Coordinates multiple sequencers in a test.",
        "chunk": "A virtual sequencer does not drive items itself; it coordinates subsequences on other sequencers in a layered testbench.",
        "document": "UVM layering: A virtual sequencer does not drive items itself; it coordinates subsequences on other sequencers in a layered testbench. Agents contain driver, sequencer, and monitor."
    },

    {
        "question": "What is functional coverage used for?",
        "answer": "Measuring which scenarios were exercised.",
        "chunk": "Functional coverage collects which stimulus/response combinations occurred to measure verification completeness beyond pass/fail.",
        "document": "Coverage: Functional coverage collects which stimulus/response combinations occurred to measure verification completeness beyond pass/fail. Code coverage complements this by measuring exercised lines/branches."
    },

    {
        "question": "Why randomize transactions in constrained-random verification?",
        "answer": "To hit corner cases not covered by directed tests.",
        "chunk": "Constrained-random generation explores the input space under constraints, often exposing corner cases missed by directed tests.",
        "document": "Stimulus: Constrained-random generation explores the input space under constraints, often exposing corner cases missed by directed tests. Constraints encode legal protocol behavior."
    },

    {
        "question": "What is a scoreboard in simulation?",
        "answer": "Compares DUT outputs to a reference model.",
        "chunk": "The scoreboard compares DUT outputs against a reference model or predicted results and flags mismatches.",
        "document": "TB architecture: The scoreboard compares DUT outputs against a reference model or predicted results and flags mismatches. Monitors feed transactions to the scoreboard."
    },
]


In [ ]:
@dataclass(frozen=True)
class AnnotatedDataPoint:
    question: str
    answer: str
    chunk: str
    document: str


def get_annotated_data_points() -> list[AnnotatedDataPoint]:
    return [AnnotatedDataPoint(**row) for row in ANNOTATED_DATA]

annotated_data = get_annotated_data_points()

dp = annotated_data[0]
print("Example AnnotatedDataPoint:")
print(f"  Question: {dp.question}")
print(f"  Answer:   {dp.answer}")
print(f"  Chunk:    {dp.chunk}")
print(f"  Document: {dp.document}")

### Corpus of Documents

`CORPUS_DOCUMENTS` is just a list of strings (multiple sentences). Consider each item to be a document parsed into string.

In [ ]:
#@title (expand to see raw data)

CORPUS_DOCUMENTS = [
    '''SVA basics: Assertions are evaluated on clocking events. In SVA, $past(expr) returns the sampled value of expr at the previous clocking event of the assertion. Related: $rose and $fell detect edges on sampled values.

Temporal operators: ##N delays by N cycles; |-> and |=> are overlapping and non-overlapping implication. Sequences can be composed with and, or, intersect.

Assertion control: The disable iff (reset) construct disables the entire assertion while reset is asserted, avoiding false failures during reset. Properties can also be aborted with procedural code.

Coverage in SVA: cover properties witness interesting traces; assume properties constrain the environment. Over-constraining can hide bugs; under-constraining can waste formal time.''',

    '''Formal setup: Use bind to instantiate a checker or SVA module alongside RTL without modifying the RTL source files. Jasper and similar tools elaborate the bound hierarchy.

Formal limits: Bounded model checking proves properties only up to a fixed depth N; deeper bugs may remain undiscovered. Increasing depth trades runtime for coverage of longer traces.

Abstraction and invariants: Cut points and black-boxing reduce state space but can introduce false failures. Invariants should be reviewed against the RTL intent.

Engine tuning: Proof strategies, case splitting, and lemma decomposition are common tactics when proofs time out or diverge.''',

    '''UVM layering: A virtual sequencer does not drive items itself; it coordinates subsequences on other sequencers in a layered testbench. Agents contain driver, sequencer, and monitor.

Stimulus: Constrained-random generation explores the input space under constraints, often exposing corner cases missed by directed tests. Constraints encode legal protocol behavior.

TB architecture: The scoreboard compares DUT outputs against a reference model or predicted results and flags mismatches. Monitors feed transactions to the scoreboard.

Coverage: Functional coverage collects which stimulus/response combinations occurred to measure verification completeness beyond pass/fail. Code coverage complements this by measuring exercised lines/branches.''',
]


## Helpers


You are also provided with this helper function to embed texts. In production, this function uses a real-embedding model -- for now, it just returns a random vector

In [ ]:
def embed_texts(
    texts: list[str],
    *,
    dim: int = 64,
    seed: int = 0,
    dummy_embedder: bool = True
) -> torch.Tensor:
    """Mock embedding API: deterministic Gaussian vectors (not text-conditioned)."""
    if dummy_embedder:
      g = torch.Generator()
      g.manual_seed(seed)
      return torch.randn(len(texts), dim, generator=g)
    else:
      from sentence_transformers import SentenceTransformer
      embed_model = SentenceTransformer("all-MiniLM-L6-v2")
    return embed_model.encode(texts, convert_to_tensor=True)


Additionally, there are also some other helper functions for training. You can view them if needed here:

In [ ]:
#@title Extra Helpers






# ── Provided helpers (not modified by candidate) ────────────────────────────

def sample_datapoints(bucket: list, m: int) -> list:
    if m <= 0 or not bucket:
        return []
    if len(bucket) >= m:
        return random.sample(bucket, m)
    return random.choices(bucket, k=m)


def create_balanced_dataset(
    rows: list,
    category_names: tuple[str, ...],
) -> Dataset:
    """Equal row counts per category; smaller buckets upsampled with replacement."""
    buckets: dict[str, list] = {c: [] for c in category_names}
    for r in rows:
        buckets[r["category"]].append(r)
    m = max(len(buckets[c]) for c in category_names)
    if m == 0:
        return Dataset.from_list([])
    balanced: list = []
    for cat in category_names:
        balanced.extend(sample_datapoints(buckets[cat], m))
    return Dataset.from_list(balanced)


def tokenize_rerank_dataset(
    ds: Dataset,
    tokenizer: PreTrainedTokenizerBase,
    *,
    question_field: str = "question",
    chunk_field: str = "chunk",
    label_field: str = "similarity_score",
) -> Dataset:
    def _tok(batch: dict[str, list]) -> dict[str, list]:
        enc = tokenizer(
            batch[question_field],
            batch[chunk_field],
            padding=False,
            truncation=True,
            max_length=256,
        )
        enc["labels"] = [float(x) for x in batch[label_field]]
        return enc

    return ds.map(_tok, batched=True, remove_columns=ds.column_names)


def make_collator(pad: DataCollatorWithPadding):
    """Returns a collate_fn that pops float labels before padding."""
    def collate(features: list[dict]) -> dict[str, torch.Tensor]:
        labels = torch.tensor([float(x.pop("labels")) for x in features], dtype=torch.float32)
        batch = pad(features)
        batch["labels"] = labels
        return batch
    return collate


### Placeholders for functions that candaite will overwrite later

# Dummy implementations — candidate will overwrite these in later sections
RerankRegressionCategory = Literal["category_1", "category_2", "category_3"]

def split_into_chunks(text: str, max_chars: int = 280) -> list[str]:
    # Candidate will overwrite this function later
    return [text[:max_chars]]


def create_regression_data_points(
    annotated_data: list[AnnotatedDataPoint],
    corpus_documents: list[str],
) -> list[RerankRegressionDataPoint]:
    # Candidate will overwrite this function later
    return []


def reranker_loss_fn(
    logits: torch.Tensor,
    labels: torch.Tensor,
    original_logits: torch.Tensor | None,
) -> torch.Tensor:
    # Candidate will overwrite this function later
    pred = logits.squeeze(-1)
    return F.mse_loss(pred, labels.float())



## Section 3: Dataset generation (15 minutes)

---

Implement the two functions below.

**`split_into_chunks`** — Split a document string into chunk strings. Each chunk should be at most `max_chars` characters.

**`create_regression_data_points`** — For each annotated datapoint, pair its question with every corpus chunk. Assign each pair a `similarity_score` (regression target) and a `category` (for balancing). You have access to `embed_texts(texts) -> Tensor` from the helpers cell.


In [ ]:
# Datatype for training datapoints
RerankRegressionCategory = Literal["positive", "negative", "hard_negative"]
class RerankRegressionDataPoint(TypedDict):
    """One training row for the cross-encoder re-ranker."""
    question: str
    chunk: str
    similarity_score: float
    category: RerankRegressionCategory

# Helper function to embed text
def embed_texts(
    texts: list[str],
    *,
    dim: int = 64,
    seed: int = 0,
    dummy_embedder: bool = True
) -> torch.Tensor:
    """Mock embedding API: deterministic Gaussian vectors (not text-conditioned)."""
    if dummy_embedder:
      g = torch.Generator()
      g.manual_seed(seed)
      return torch.randn(len(texts), dim, generator=g)
    else:
      from sentence_transformers import SentenceTransformer
      embed_model = SentenceTransformer("all-MiniLM-L6-v2")
    return embed_model.encode(texts, convert_to_tensor=True)



#########################################################################################
# TODO: Implement this function
#########################################################################################
def split_into_chunks(text: str, max_chars: int = 280) -> list[str]:
    ## Function that splits a document into chunks of at most max_chars characters.
    ## To be used by create_regression_data_points()
    chunks = []
    for i in range(0, len(text), max_chars):
        chunks.append(text[i : i + max_chars])
    return chunks
    return []
#########################################################################################


def create_regression_data_points(
    annotated_data: list[AnnotatedDataPoint],
    corpus_documents: list[str],
) -> list[RerankRegressionDataPoint]:
    """
    Return a list of RerankRegressionDataPoint dicts, each with:
      - question, chunk, similarity_score, category
    Use split_into_chunks() to split corpus documents into chunks.
    You can use `embed_texts(texts)` to get embeddings.
    """

    # Extract chunks from documents
    #chunk_texts = [ch for doc in corpus_documents for ch in split_into_chunks(doc)]
    # Extract chunks from documents using itertools for better performance
    from itertools import chain
    chunk_texts = list(chain.from_iterable(split_into_chunks(doc) for doc in corpus_documents))
    # Each row is a dict(RerankRegressionDataPoint)
    #########################################################################################
    # TODO: Implement logic to build regression datapoints from annotated_data and chunk_texts
    chunk_embeddings = embed_texts(chunk_texts)
    rows: list[RerankRegressionDataPoint] = []
    for dp in annotated_data:
      rows.append({
          "question": dp.question,
          "chunk": dp.chunk,
          "similarity_score": 1,
          "category": "positive"
      })
      q_enc = embed_texts([dp.question])
      score = F.cosine_similarity(q_enc, chunk_embeddings,dim=1)
      top_indices = torch.argsort(score,descending=True)[:5]
      for idx in top_indices:
        candidate_chunk = chunk_texts[idx]
        rows.append({
            "question": dp.question,
            "chunk": candidate_chunk,
            "similarity_score": score[idx].item(),
            "category": "hard_negative"
        })
        random_chunks = random.sample(chunk_texts, 3)
        for chunk in random_chunks:
          rows.append({
              "question": dp.question,
              "chunk": chunk,
              "similarity_score":0.0,
              "category": "negative"
          })
    #########################################################################################
    #rows: list[Dict[str, Any]] = []


    return rows

#########################################################################################


## Section 4: Loss function (10 minutes)

### The training flow


Using the dataset you constructed, we can now train a re-ranker model.
Instead of training from scratch, we will tune an existing re-ranking model.

The loss below already has supervised MSE (model prediction vs. `similarity_score` label). Since we are tuning an existing re-ranking model, update this loss function to prevent the trained model from drifting too much from the original model.


In [ ]:
def reranker_loss_fn(
    logits: torch.Tensor,
    labels: torch.Tensor,
    original_logits: torch.Tensor | None,
) -> torch.Tensor:
    """
    `original_logits` contains the frozen original model's output for the same batch
    """
    pred = logits.squeeze(-1)
    targets = labels.float()
    task_loss = F.mse_loss(pred, targets)

    ############################################################################
    ## TODO: Modify the loss function to minimize drfit from original model

    if original_logits is not None:
        # 2. KL Divergence term
        # We treat the scores in the batch as a distribution.
        # T (Temperature) controls how much we smooth the distribution.
        T = 2.0

        # Original model (Teacher) distribution
        # Logits are scaled by T to soften the probability distribution
        p_teacher = F.softmax(original_logits.squeeze(-1) / T, dim=-1)

        # Current model (Student) log-distribution
        log_p_student = F.log_softmax(pred / T, dim=-1)

        # KL Divergence: How much does Student vary from Teacher?
        # We multiply by T^2 as per standard distillation practices
        distillation_loss = F.kl_div(log_p_student, p_teacher, reduction='batchmean') * (T**2)

        # 3. Combined Loss
        # alpha = weight given to the original model's behavior
        alpha = 0.4
        loss = (1 - alpha) * task_loss + (alpha) * distillation_loss
    else:
        loss = task_loss

    return loss


We can now run training!

In [ ]:
class RerankRegressionTrainer(Trainer):
    """Feeds the frozen original model's logits into the loss for the drift term."""

    def __init__(
        self,
        loss_fn: Callable,
        original_model: Optional[PreTrainedModel] = None,
        *args: object,
        **kwargs: object,
    ) -> None:
        super().__init__(*args, **kwargs)
        self.original_model = original_model
        self.loss_fn = loss_fn

    def compute_loss(
        self,
        model: PreTrainedModel,
        inputs: dict[str, torch.Tensor],
        return_outputs: bool = False,
        num_items_in_batch: torch.Tensor | None = None,
    ) -> torch.Tensor | tuple[torch.Tensor, object]:
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        original_logits: torch.Tensor | None = None
        if self.original_model is not None:
            self.original_model.eval()
            with torch.no_grad():
                original_logits = self.original_model(**inputs).logits
        loss = self.loss_fn(logits, labels, original_logits)
        return (loss, outputs) if return_outputs else loss

def main():
    # 1. Build training rows from annotated data + corpus
    ###### Candidate implemented this ######
    datarows = create_regression_data_points(annotated_data, CORPUS_DOCUMENTS)
    data_categories = get_args(RerankRegressionCategory)
    #######################################
    balanced_training_dataset = create_balanced_dataset(datarows, data_categories)

    # 2. Tokenize for the cross-encoder
    MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_tok = tokenize_rerank_dataset(balanced_training_dataset, tokenizer)
    _pad = DataCollatorWithPadding(tokenizer)

    # 3. Load two copies of the same checkpoint
    rerank_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
    original_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
    original_model.eval()
    for p in original_model.parameters():
        p.requires_grad_(False)


    # 4. Define loss function
    ###### Candidate implements this ######
    loss_fn = reranker_loss_fn
    #######################################

    # 5. Train with drift-aware loss
    trainer = RerankRegressionTrainer(
        model=rerank_model,
        args=TrainingArguments(
            output_dir="./rerank_mse_out",
            num_train_epochs=3,
            per_device_train_batch_size=4,
            logging_steps=5,
            report_to="none",
            remove_unused_columns=False,
        ),
        loss_fn=loss_fn,
        train_dataset=train_tok,
        processing_class=tokenizer,
        data_collator=make_collator(_pad),
        original_model=original_model,
    )
    trainer.train()

main()